# Assignment 1 — training pipeline

Predicts whether a speed date ends in a **mutual match** (`match` = both said
yes), for the organiser scenario in `REPORT.md`: predictions are made the
morning after the event, so profile information and scorecard ratings may be
used, but the yes/no decisions may not.

Sections:
1. Setup and data loading
2. Leakage check and column roles
3. Event-grouped train / validation / test split
4. Evaluation helpers (ROC-AUC, PR-AUC, business cost, threshold choice)
5. Naive baseline
6. Feature engineering and grouped-CV feature selection
7. One model, three implementations: scikit-learn, manual PyTorch, standard PyTorch
8. Comparison: agreement between implementations, test results vs. baseline
9. Saving artifacts for the app

## 1. Setup and data loading

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score

SEED = 42
DATA_PATH = Path("data/speed_dating_master.csv")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

# Business cost from REPORT.md: a missed match (FN) costs 3x a wasted nudge (FP).
COST_FP = 1
COST_FN = 3

np.random.seed(SEED)

df = pd.read_csv(DATA_PATH)
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Events: {df['event_id'].nunique()}, missing values: {df.isna().sum().sum()}")
print(f"Match rate: {df['match'].mean():.3f}")

## 2. Leakage check and column roles

`match` is computed from the two individual decisions, so they would hand the
model the answer. The check below confirms that on every row before dropping
them. The participant IDs are kept for now (they may be useful for
per-rater feature engineering) but are never used as model inputs.

In [ ]:
TARGET = "match"
LEAKAGE_COLS = ["male_decision", "female_decision"]
ID_COLS = ["event_id", "male_id", "female_id"]

assert (df[TARGET] == (df["male_decision"] & df["female_decision"])).all(), \
    "match is not simply male_decision AND female_decision"
df = df.drop(columns=LEAKAGE_COLS)

candidate_features = [c for c in df.columns if c not in ID_COLS + [TARGET]]
print(f"{len(candidate_features)} candidate feature columns:")
print(candidate_features)

## 3. Event-grouped train / validation / test split

Every participant attends exactly one event and has 6–15 dates there. Splitting
whole events (about 70 / 15 / 15) keeps each person in only one split, so the
test score measures performance on *new people*, not memorised individuals.
There are no dates, so a time-based split isn't possible. The validation set
is used to choose the threshold and tune hyperparameters; the test set is used
once, at the end.

In [ ]:
rng = np.random.default_rng(SEED)
events = rng.permutation(df["event_id"].unique())
n_train = int(0.70 * len(events))
n_val = int(0.15 * len(events))

split_of_event = {e: "train" for e in events[:n_train]}
split_of_event.update({e: "val" for e in events[n_train:n_train + n_val]})
split_of_event.update({e: "test" for e in events[n_train + n_val:]})
df["split"] = df["event_id"].map(split_of_event)

# No participant may appear in more than one split.
for id_col in ["male_id", "female_id"]:
    assert df.groupby(id_col)["split"].nunique().max() == 1

print(df.groupby("split").agg(
    events=("event_id", "nunique"),
    rows=(TARGET, "size"),
    match_rate=(TARGET, "mean"),
).loc[["train", "val", "test"]].round(3))

## 4. Evaluation helpers

* **ROC-AUC / PR-AUC** compare models independently of any threshold.
* **Business cost** = FP × 1 + FN × 3 at a given threshold.
* The threshold is chosen to minimise business cost on the **validation** set,
  then applied unchanged to the test set.

In [ ]:
def business_cost(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return COST_FP * fp + COST_FN * fn


def best_threshold(y_true, y_prob, grid=np.linspace(0, 1, 101)):
    costs = [business_cost(y_true, y_prob, t) for t in grid]
    return float(grid[int(np.argmin(costs))])


def evaluate(name, y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": name,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "threshold": threshold,
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "business_cost": int(COST_FP * fp + COST_FN * fn),
        "cost_per_date": (COST_FP * fp + COST_FN * fn) / len(y_true),
    }

## 5. Naive baseline

The baseline ignores every feature and gives each date the training match
rate as its probability. Any real model must beat it. Because all its
probabilities are equal, the best threshold can only be "nudge everyone" or
"nudge no one"; with matches at ~20% and FN costing 3x, the cheaper option
should be "nudge no one".

In [ ]:
y_train = df.loc[df["split"] == "train", TARGET].to_numpy()
y_val = df.loc[df["split"] == "val", TARGET].to_numpy()
y_test = df.loc[df["split"] == "test", TARGET].to_numpy()

base_rate = y_train.mean()
base_val = np.full(len(y_val), base_rate)
base_test = np.full(len(y_test), base_rate)

# Constant predictions: only "everyone" (t <= base_rate) or "no one" (t > base_rate) differ.
base_threshold = min([0.0, 1.0], key=lambda t: business_cost(y_val, base_val, t))
baseline = evaluate("Naive baseline", y_test, base_test, base_threshold)

results = [baseline]
print(pd.DataFrame(results).round(3).to_string(index=False))

## 6. Feature engineering

Five decisions (reasons in `REPORT.md`):

1. **"Both interested" features** — `min()` of the two ratings for each
   attribute, plus the product for attractiveness and for each person's
   overall rating. Logistic regression only adds its inputs, so it can't
   express "both must agree" without these.
2. **Rater calibration** — *tested, then dropped*: each rating minus the
   rater's own average rating for that attribute across all their dates
   (ratings only, never decisions). Grouped CV showed it slightly lowered
   PR-AUC, so it is kept only if it raises mean CV PR-AUC.
3. **Preference-weighted ratings** — *experiment*: kept only if they improve
   validation PR-AUC.
4. **`goes_out`** — ordered 0–3; one-hot only if validation shows a clear
   gain.
5. **Profile features** — kept, with an ablation (profile-only vs. all
   features) to measure how much they contribute.

Some of the "who rated whom" naming: `attr_of_female` is the attractiveness
the **male** gave the **female**, so the male is the rater.

In [ ]:
ATTRS = ["attr", "sinc", "intel", "fun", "amb"]
GOES_OUT_ORDER = {"rarely": 0, "sometimes": 1, "often": 2, "very_often": 3}

# Which participant gave each rating column.
RATER_OF = {f"{a}_of_female": "male_id" for a in ATTRS}
RATER_OF.update({f"{a}_of_male": "female_id" for a in ATTRS})

fe = df.copy()

# 4. goes_out as ordered numbers (one-hot version built separately for the experiment).
for who in ["male", "female"]:
    fe[f"{who}_goes_out_ord"] = fe[f"{who}_goes_out"].map(GOES_OUT_ORDER)
assert fe[["male_goes_out_ord", "female_goes_out_ord"]].notna().all().all()

# 1. "Both interested": a match needs both people, so model the weaker side.
for a in ATTRS:
    fe[f"min_{a}"] = fe[[f"{a}_of_female", f"{a}_of_male"]].min(axis=1)
fe["overall_by_male"] = fe[[f"{a}_of_female" for a in ATTRS]].mean(axis=1)
fe["overall_by_female"] = fe[[f"{a}_of_male" for a in ATTRS]].mean(axis=1)
fe["min_overall"] = fe[["overall_by_male", "overall_by_female"]].min(axis=1)
fe["prod_attr"] = fe["attr_of_female"] * fe["attr_of_male"]
fe["prod_overall"] = fe["overall_by_male"] * fe["overall_by_female"]

# 2. Rater calibration: rating relative to the rater's own average.
# Each person attends one event and so belongs to exactly one split:
# these averages never mix information across train / val / test.
for col, rater in RATER_OF.items():
    fe[f"{col}_rel"] = fe[col] - fe.groupby(rater)[col].transform("mean")
for col, rater in [("overall_by_male", "male_id"), ("overall_by_female", "female_id")]:
    fe[f"{col}_rel"] = fe[col] - fe.groupby(rater)[col].transform("mean")
fe["min_overall_rel"] = fe[["overall_by_male_rel", "overall_by_female_rel"]].min(axis=1)

# 3. Preference-weighted ratings (experiment): stated weight (0-100 pts) x rating given.
fe["male_wtd_attr"] = fe["male_pref_attr"] / 100 * fe["attr_of_female"]
fe["male_wtd_intel"] = fe["male_pref_intel"] / 100 * fe["intel_of_female"]
fe["female_wtd_attr"] = fe["female_pref_attr"] / 100 * fe["attr_of_male"]
fe["female_wtd_intel"] = fe["female_pref_intel"] / 100 * fe["intel_of_male"]

# One-hot goes_out (only for the encoding experiment).
goes_out_onehot = pd.get_dummies(
    fe[["male_goes_out", "female_goes_out"]], dtype=float
)
fe = pd.concat([fe, goes_out_onehot], axis=1)

# Feature groups.
PROFILE = ["male_age", "female_age", "age_gap", "same_race", "same_field",
           "male_pref_attr", "male_pref_intel", "female_pref_attr", "female_pref_intel",
           "male_self_attr", "female_self_attr"]
GOES_OUT_ORD = ["male_goes_out_ord", "female_goes_out_ord"]
GOES_OUT_ONEHOT = list(goes_out_onehot.columns)
RATINGS = ["shared_interests"] + list(RATER_OF)
BOTH_INTERESTED = [f"min_{a}" for a in ATTRS] + [
    "overall_by_male", "overall_by_female", "min_overall", "prod_attr", "prod_overall"]
CALIBRATED = [f"{c}_rel" for c in RATER_OF] + [
    "overall_by_male_rel", "overall_by_female_rel", "min_overall_rel"]
PREF_WEIGHTED = ["male_wtd_attr", "male_wtd_intel", "female_wtd_attr", "female_wtd_intel"]

print(f"Engineered table: {fe.shape[1]} columns")

### Feature experiments (grouped cross-validation on training events)

The validation split is only 18 events, so small PR-AUC differences there are
mostly noise. Instead, each feature set is scored with **5-fold
cross-validation grouped by event** on the 84 training events: every fold
holds out whole events, the scaler is refitted inside each fold, and we report
the mean and standard deviation of PR-AUC across folds. Validation and test
are not touched here.

The decision rules from `REPORT.md` are applied in code:

* rater-calibrated features are kept only if they raise mean CV PR-AUC
  (they did not, so they were removed);
* preference-weighted features are kept only if they raise mean CV PR-AUC;
* one-hot `goes_out` replaces ordinal only if it gains at least
  `CLEAR_GAIN` mean CV PR-AUC.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CLEAR_GAIN = 0.005  # minimum mean PR-AUC improvement that counts as "clear"
N_FOLDS = 5

train_mask = fe["split"] == "train"
val_mask = fe["split"] == "val"
test_mask = fe["split"] == "test"
train_df = fe.loc[train_mask]
cv = GroupKFold(n_splits=N_FOLDS)


def cv_scores(cols, C=1.0):
    """Grouped 5-fold CV on training events: mean/std ROC-AUC and PR-AUC."""
    model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
    s = cross_validate(
        model, train_df[cols], train_df[TARGET],
        groups=train_df["event_id"], cv=cv,
        scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"},
    )
    return (s["test_roc_auc"].mean(), s["test_pr_auc"].mean(), s["test_pr_auc"].std())


BASE = PROFILE + GOES_OUT_ORD + RATINGS
core = BASE + BOTH_INTERESTED
experiments = {
    "Profile only (+ goes_out)": PROFILE + GOES_OUT_ORD,
    "Raw ratings only": RATINGS,
    "Profile + raw ratings (no engineering)": BASE,
    "core: + both-interested": core,
    "core + calibrated": core + CALIBRATED,
    "core + preference-weighted": core + PREF_WEIGHTED,
    "core + calibrated + preference-weighted": core + CALIBRATED + PREF_WEIGHTED,
    "core with one-hot goes_out": [c for c in core if c not in GOES_OUT_ORD] + GOES_OUT_ONEHOT,
}

ablation = pd.DataFrame(
    [(name, len(cols), *cv_scores(cols)) for name, cols in experiments.items()],
    columns=["feature set", "n_features", "cv_roc_auc", "cv_pr_auc", "cv_pr_auc_std"],
).set_index("feature set")
print(ablation.round(4).to_string())

core_pr = ablation.loc["core: + both-interested", "cv_pr_auc"]
keep_calibrated = ablation.loc["core + calibrated", "cv_pr_auc"] > core_pr
keep_pref_weighted = ablation.loc["core + preference-weighted", "cv_pr_auc"] > core_pr
use_onehot = ablation.loc["core with one-hot goes_out", "cv_pr_auc"] - core_pr >= CLEAR_GAIN

FEATURES = list(core)
if keep_calibrated:
    FEATURES += CALIBRATED
if keep_pref_weighted:
    FEATURES += PREF_WEIGHTED
if use_onehot:
    FEATURES = [c for c in FEATURES if c not in GOES_OUT_ORD] + GOES_OUT_ONEHOT

print(f"\nKeep rater-calibrated features: {keep_calibrated}")
print(f"Keep preference-weighted features: {keep_pref_weighted}")
print(f"Use one-hot goes_out: {use_onehot}")
print(f"Final feature set: {len(FEATURES)} features")
ablation.to_csv(ARTIFACTS / "feature_ablation.csv")

### Final design matrices

The scaler is fitted on the **training split only** and then applied to
validation and test, so no information from those splits leaks into the
preprocessing. The same scaled arrays feed all three models.

In [ ]:
scaler = StandardScaler().fit(fe.loc[train_mask, FEATURES])

X_train = scaler.transform(fe.loc[train_mask, FEATURES]).astype(np.float32)
X_val = scaler.transform(fe.loc[val_mask, FEATURES]).astype(np.float32)
X_test = scaler.transform(fe.loc[test_mask, FEATURES]).astype(np.float32)
print("X_train", X_train.shape, "X_val", X_val.shape, "X_test", X_test.shape)

## 7. One model, three implementations

All three are **L2-regularised logistic regression** trained on the same 38
standardised features (`X_train`) and judged on the same validation and test
rows.

**Making them comparable.** scikit-learn minimises
`0.5·||w||² + C·Σ log-loss`. Dividing by `C·n` gives the equivalent
per-sample objective `mean log-loss + (λ/2)·||w||²` with `λ = 1/(C·n_train)`,
which is what both PyTorch versions minimise. The intercept is not penalised
in any of them.

**Tuning (sparingly).** `C` is picked by the same grouped 5-fold CV as the
feature experiments. The learning rate for the two PyTorch versions is picked
from a short grid as the one that reaches the lowest training objective
(full-batch gradient descent, fixed number of epochs).

### 7a. scikit-learn

In [ ]:
import joblib
import torch
import torch.nn as nn

C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]
c_search = pd.DataFrame(
    [(C, *cv_scores(FEATURES, C=C)) for C in C_GRID],
    columns=["C", "cv_roc_auc", "cv_pr_auc", "cv_pr_auc_std"],
)
print(c_search.round(4).to_string(index=False))
BEST_C = float(c_search.loc[c_search["cv_pr_auc"].idxmax(), "C"])
print(f"Chosen C = {BEST_C}")

sk_model = LogisticRegression(C=BEST_C, max_iter=5000, tol=1e-8)
sk_model.fit(X_train, y_train)

LAMBDA = 1.0 / (BEST_C * len(X_train))
print(f"Equivalent PyTorch L2 strength lambda = {LAMBDA:.6f}")

### 7b. Manual PyTorch loop

Raw tensors `w` and `b` with `requires_grad=True`. Each epoch computes the
loss by hand (sigmoid → binary cross-entropy → plus L2 penalty), calls
`backward()` for the gradients, then updates the parameters manually inside
`torch.no_grad()` and resets the gradients.

In [ ]:
Xt_train = torch.from_numpy(X_train)
yt_train = torch.from_numpy(y_train.astype(np.float32))
Xt_val = torch.from_numpy(X_val)
Xt_test = torch.from_numpy(X_test)
N_FEATURES = X_train.shape[1]

EPOCHS = 3000
LR_GRID = [0.05, 0.1, 0.5, 1.0]


def train_manual(lr, epochs=EPOCHS):
    torch.manual_seed(SEED)
    w = (torch.randn(N_FEATURES) * 0.01).requires_grad_()
    b = torch.zeros(1, requires_grad=True)
    losses = []
    for _ in range(epochs):
        logits = Xt_train @ w + b
        p = torch.sigmoid(logits).clamp(1e-7, 1 - 1e-7)
        bce = -(yt_train * torch.log(p) + (1 - yt_train) * torch.log(1 - p)).mean()
        loss = bce + 0.5 * LAMBDA * (w ** 2).sum()
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
        losses.append(loss.item())
    return w.detach(), b.detach(), losses


manual_runs = {lr: train_manual(lr) for lr in LR_GRID}
final_loss = {lr: (run[2][-1] if np.isfinite(run[2][-1]) else np.inf) for lr, run in manual_runs.items()}
for lr, l in final_loss.items():
    print(f"lr={lr:<5} final training objective = {l:.6f}")
MANUAL_LR = min(final_loss, key=final_loss.get)
manual_w, manual_b, manual_losses = manual_runs[MANUAL_LR]
print(f"Chosen learning rate (manual) = {MANUAL_LR}")


def manual_predict(X):
    return torch.sigmoid(X @ manual_w + manual_b).numpy()

### 7c. Standard PyTorch workflow

An `nn.Module` with one `nn.Linear` layer, `nn.BCEWithLogitsLoss` (which fuses
the sigmoid and the log-loss for numerical stability) and `torch.optim.SGD`.
The L2 penalty is added explicitly to the **weights only**: SGD's built-in
`weight_decay` would also shrink the bias, which scikit-learn doesn't do.

In [ ]:
class LogisticRegressionModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)  # logits


def train_standard(lr, epochs=EPOCHS):
    torch.manual_seed(SEED)
    model = LogisticRegressionModel(N_FEATURES)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    losses = []
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(Xt_train), yt_train) + 0.5 * LAMBDA * (model.linear.weight ** 2).sum()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return model, losses


standard_runs = {lr: train_standard(lr) for lr in LR_GRID}
final_loss_std = {lr: (run[1][-1] if np.isfinite(run[1][-1]) else np.inf) for lr, run in standard_runs.items()}
for lr, l in final_loss_std.items():
    print(f"lr={lr:<5} final training objective = {l:.6f}")
STANDARD_LR = min(final_loss_std, key=final_loss_std.get)
std_model, standard_losses = standard_runs[STANDARD_LR]
std_model.eval()
print(f"Chosen learning rate (standard) = {STANDARD_LR}")


def standard_predict(X):
    with torch.no_grad():
        return torch.sigmoid(std_model(X)).numpy()

## 8. Comparison

### 8a. Do the three implementations agree?

Same model + same data + same objective should give (almost) the same
coefficients and probabilities. Differences would come from the optimiser
stopping before the exact optimum (gradient descent with a fixed number of
epochs) rather than from the model itself.

In [ ]:
def sk_objective(w, b):
    p = np.clip(1 / (1 + np.exp(-(X_train @ w + b))), 1e-7, 1 - 1e-7)
    bce = -np.mean(y_train * np.log(p) + (1 - y_train) * np.log(1 - p))
    return bce + 0.5 * LAMBDA * np.sum(w ** 2)


coefs = pd.DataFrame({
    "feature": FEATURES,
    "sklearn": sk_model.coef_.ravel(),
    "manual_torch": manual_w.numpy(),
    "standard_torch": std_model.linear.weight.detach().numpy().ravel(),
})
intercepts = {
    "sklearn": float(sk_model.intercept_[0]),
    "manual_torch": float(manual_b[0]),
    "standard_torch": float(std_model.linear.bias.detach()[0]),
}
objective = {
    "sklearn": sk_objective(sk_model.coef_.ravel(), sk_model.intercept_[0]),
    "manual_torch": sk_objective(manual_w.numpy(), float(manual_b[0])),
    "standard_torch": sk_objective(coefs["standard_torch"].to_numpy(), intercepts["standard_torch"]),
}

probs_test = {
    "Naive baseline": base_test,
    "scikit-learn": sk_model.predict_proba(X_test)[:, 1],
    "Manual PyTorch": manual_predict(Xt_test),
    "Standard PyTorch": standard_predict(Xt_test),
}
probs_val = {
    "Naive baseline": base_val,
    "scikit-learn": sk_model.predict_proba(X_val)[:, 1],
    "Manual PyTorch": manual_predict(Xt_val),
    "Standard PyTorch": standard_predict(Xt_val),
}

print("Training objective (lower = closer to the optimum):")
for k, v in objective.items():
    print(f"  {k:<15} {v:.6f}")
print("Intercepts:", {k: round(v, 4) for k, v in intercepts.items()})
for other in ["manual_torch", "standard_torch"]:
    diff = (coefs[other] - coefs["sklearn"]).abs()
    print(f"sklearn vs {other}: max |coef diff| = {diff.max():.4f} "
          f"(feature: {coefs.loc[diff.idxmax(), 'feature']}), mean = {diff.mean():.4f}")
for name in ["Manual PyTorch", "Standard PyTorch"]:
    d = np.abs(probs_test[name] - probs_test["scikit-learn"])
    print(f"Test probabilities, sklearn vs {name}: max |diff| = {d.max():.5f}, mean = {d.mean():.5f}")

coefs["abs_sklearn"] = coefs["sklearn"].abs()
print("\nLargest coefficients (standardised features):")
print(coefs.sort_values("abs_sklearn", ascending=False).drop(columns="abs_sklearn").head(12).round(4).to_string(index=False))

### 8b. Test-set results against the baseline

Each model's threshold is chosen to minimise business cost on **validation**,
then applied unchanged to the **test** set (used only here, once).

In [ ]:
thresholds = {"Naive baseline": base_threshold}
for name in ["scikit-learn", "Manual PyTorch", "Standard PyTorch"]:
    thresholds[name] = best_threshold(y_val, probs_val[name])

results = [evaluate(name, y_test, probs_test[name], thresholds[name]) for name in probs_test]
for r in results:
    r["val_pr_auc"] = average_precision_score(y_val, probs_val[r["model"]])
results_df = pd.DataFrame(results).set_index("model")
print(results_df[["val_pr_auc", "roc_auc", "pr_auc", "threshold", "tp", "fp", "fn", "tn",
                  "business_cost", "cost_per_date"]].round(3).to_string())

saving = 1 - results_df.loc["scikit-learn", "business_cost"] / results_df.loc["Naive baseline", "business_cost"]
print(f"\nscikit-learn cuts business cost by {saving:.1%} vs. the baseline on test.")

## 9. Save artifacts for the app

The app must never retrain, so everything it needs is saved here: the three
fitted models, the fitted scaler, the feature list, the engineered table (for
distribution plots and re-scoring), loss curves, coefficients and metrics.

In [ ]:
joblib.dump(sk_model, ARTIFACTS / "sklearn_model.joblib")
joblib.dump(scaler, ARTIFACTS / "scaler.joblib")
torch.save({"w": manual_w, "b": manual_b}, ARTIFACTS / "manual_torch.pt")
torch.save(std_model.state_dict(), ARTIFACTS / "standard_torch.pt")

fe[ID_COLS + ["split", TARGET] + FEATURES].to_csv(ARTIFACTS / "features_table.csv", index=False)
df[ID_COLS + ["split"]].to_csv(ARTIFACTS / "split_assignment.csv", index=False)
coefs.drop(columns="abs_sklearn").to_csv(ARTIFACTS / "coefficients.csv", index=False)
pd.DataFrame({"epoch": range(1, EPOCHS + 1), "manual_torch": manual_losses,
              "standard_torch": standard_losses}).to_csv(ARTIFACTS / "loss_curves.csv", index=False)

with open(ARTIFACTS / "metrics.json", "w") as f:
    json.dump({
        "cost_fp": COST_FP, "cost_fn": COST_FN,
        "features": FEATURES, "best_C": BEST_C, "lambda": LAMBDA,
        "manual_lr": MANUAL_LR, "standard_lr": STANDARD_LR, "epochs": EPOCHS,
        "thresholds": thresholds, "intercepts": intercepts,
        "training_objective": objective,
        "results": results_df.reset_index().to_dict(orient="records"),
    }, f, indent=2, default=float)
c_search.to_csv(ARTIFACTS / "c_search.csv", index=False)
print("Saved:", sorted(p.name for p in ARTIFACTS.iterdir()))